In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import Any, cast
import os
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import numpy as np
import torch
import sys
sys.path.append(str(Path(os.getcwd()).resolve().parent.parent))
from metaworld_dataset import (
    MetaworldDataModule,
    color_blind_visual_domain,
    get_default_domains,
)
from tqdm import tqdm

from shimmer_metaworld import DEBUG_MODE, PROJECT_DIR
from shimmer_metaworld.config import DomainModuleVariant, load_config
from shimmer_metaworld.modules.domains.pretrained import load_pretrained_module
from shimmer_metaworld.modules.domains.visual import VisualDomainModule



NameError: name '__file__' is not defined

**Loading and pickling model variables**

In [ ]:
import pickle

config = load_config(
        PROJECT_DIR / "shimmer_metaworld/config_template",
        load_files=["save_v_latents.yaml"],
        debug_mode=DEBUG_MODE,
    )
beta = 1
additional_transforms: dict[str, list[Callable[[Any], Any]]] = {}
if config.domain_modules.visual.color_blind:
    additional_transforms["v"] = [color_blind_visual_domain]

data_module = MetaworldDataModule(
    os.path.abspath(config.dataset.path),
    get_default_domains(["v"]),
    {frozenset(["v"]): 1.0},
    batch_size=config.training.batch_size,
    num_workers=config.training.num_workers,
    seed=config.seed,
    additional_transforms=additional_transforms,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
domain_checkpoint = None
for domain in config.domains:
    print(domain.domain_type)
    if domain.domain_type == DomainModuleVariant.v:
        domain_checkpoint = domain

assert (
    domain_checkpoint is not None
), "Please add domain_checkpoint entry in the configuration"
assert domain_checkpoint.domain_type == DomainModuleVariant.v
print(domain_checkpoint.checkpoint_path)
visual_domain = cast(
    VisualDomainModule,
    load_pretrained_module(domain_checkpoint),
)
visual_domain.to(device)
visual_domain.freeze()

data_module.prepare_data()
data_module.setup()

DomainModuleVariant.v
/mnt/datashare/yelhelw/checkpoints/shimmer-meta-c8fv9fqk/epoch=14.ckpt


In [18]:
dataloaders = {
    "val": data_module.val_dataloader(),
}
for split, dataloader in dataloaders.items():
    latents: list[np.ndarray] = []

    print(f"Saving {split}.")
    for batch, _, _ in tqdm(iter(dataloader), total=len(dataloader)):
        if split == "train":
            images = batch[frozenset(["v"])]["v"].to(device)
        else:
            images = batch[frozenset(["v"])]["v"].to(device)
        latent = visual_domain.encode(images)
        latents.append(latent.detach().cpu().numpy())

    latent_vectors = np.concatenate(latents, axis=0)
    shuffle_latent_vectors = latent_vectors.copy()
    np.random.shuffle(shuffle_latent_vectors)

print(len(latent_vectors))

Saving val.


100%|██████████| 98/98 [00:31<00:00,  3.14it/s]


100050


__Latent UMAP structures__

In [13]:
from collections.abc import Mapping
from shimmer.modules.selection import FixedSharedSelection
from matplotlib.colors import ListedColormap
from tqdm import tqdm


def to_device(data: torch.Tensor | Mapping[str, torch.Tensor] | list, device: str):
    """Put the data Tensor or list on the device (GPU or CPU)"""
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, list):
        return [value.to(device) for value in data]
    elif isinstance(data, Mapping):
        return {name: to_device(value, device) for name, value in data.items()}
    else:
        raise TypeError(f"Unsupported type: {type(data)}")


<p align="center">
    UMAP parameters sweep
</p>

In [ ]:
from umap.umap_ import nearest_neighbors
import pickle



knn = nearest_neighbors(shuffle_latent_vectors,
                              n_neighbors=100,
                              metric="cosine",
                              metric_kwds=None,
                              angular=True,
                              random_state=None,
                             )


In [ ]:
import umap

n_neighbors = [15, 25,50, 100]
min_dists = [0.1,0.2, 0.5, 0.9]
embeddings = np.zeros((4, 4, 100047, 2))
for i, k in enumerate(n_neighbors):
    for j, dist in enumerate(min_dists):
        print(k,dist)
        reducer = umap.UMAP(n_neighbors=k, verbose=True,
                                                      min_dist=dist,
                                                      ).fit(shuffle_latent_vectors)
                                                      
        print("reducer done")
        embeddings[i, j] = reducer.transform(latent_vectors)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(4, 4, figsize=(20, 20))

for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        for obj_label, color in color_map.items():
                mask = obj_code == obj_label
                ax.scatter(embeddings[i,j,mask, 0],
                embeddings[i,j,mask, 1],
                c=color,
                alpha=0.8,
                s=5,
                label=obj_label,
                )

        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title("min_dist = {}".format(min_dists[j]), size=15)
        if j == 0:
            ax.set_ylabel("n_neighbors = {}".format(n_neighbors[i]), size=15)
fig.suptitle("UMAP embedding of vision VAE latents with grid of parameters", y=0.92, size=20)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.savefig("graphs/vae/UMAP_sweep.png")
plt.close()

In [6]:
labels = np.load(f"{config.dataset.path}/actions_val.npy", mmap_mode="r")
labels_train = np.clip(np.load(f"{config.dataset.path}/actions_train.npy", mmap_mode="r"), -10, 10)
all_labels = np.repeat(labels, 3, axis=0)

attributes = np.load(f"{config.dataset.path}/attributes_val.npy", mmap_mode="r")
print(len(attributes))
wall = attributes[:,9]
ball = attributes[:,6]
goal = attributes[:,12]
print(len(latent_vectors))
print(np.where(ball[attributes[:,-1]==-1]==0))
modality_names = {0: 'Vision (v)', 1: 'Attributes (attr)', 2: 'Actions (act)'}

print(np.where(attributes[:len(labels),5] < 0))
labels_norm = np.clip(labels.copy(), -1, 1)

#labels_norm = labels.copy()
#action_mean = np.mean(labels_train,axis=0)
#action_stdv = np.std(labels_train,axis=0)
#for x in range(4):
#    labels_norm[:,x] = (labels_norm[:,x]-action_mean[x])/ action_stdv[x]
#    labels_norm[:,x] = np.clip(labels_norm[:,x],-1,1)
x_disp = labels_norm[:, 0]
y_disp = labels_norm[:, 1]
z_disp = labels_norm[:, 2]
gripper = labels_norm[:, 3]

print("Transforming latent vectors...")
act_shuffle = np.array(labels_norm.copy())
np.random.shuffle(act_shuffle)


100050
100050
(array([], dtype=int64),)
(array([], dtype=int64),)
Transforming latent vectors...


In [7]:
wall_binary = np.where((wall==-10),0,1)
print(np.where(wall==-10)[0])
x_disp_binary = np.where((x_disp<0),0,1)
goal_binary = np.where((goal==-10),0,1)
ball_binary = np.where((ball==-10),0,1)
print(np.where(ball==-10)[0])

# Create obj_code with different combinations
obj_code = np.zeros(len(wall_binary), dtype=object)
obj_code[:] = "none"

# all - wall AND goal AND ball
mask_all = (wall_binary == 1) & (goal_binary == 1) & (ball_binary == 1)
obj_code[mask_all] = "all"

# wall_ball - wall AND ball (goal False)
mask_wall_ball = (wall_binary == 1) & (ball_binary == 1) & (goal_binary == 0)
obj_code[mask_wall_ball] = "wall_ball"

# wall - only wall
mask_wall_only = (wall_binary == 1) & (goal_binary == 0) & (ball_binary == 0)
obj_code[mask_wall_only] = "wall"

# goal_ball - goal AND ball (wall False)
mask_goal_ball = (goal_binary == 1) & (ball_binary == 1) & (wall_binary == 0)
obj_code[mask_goal_ball] = "goal_ball"

color_map = {
    #"none": '#1f77b4',
    "all": '#ff7f0e',
    "wall_ball": '#2ca02c',
    "wall": '#d62728',
    "goal_ball": '#9467bd'
}

[40011 40012 40013 ... 60012 60013 60014]
[20011 20012 20013 ... 40008 40009 40010]


In [18]:
import umap
reducer = umap.UMAP(n_neighbors=50,min_dist=0.5,metric="cosine",metric_kwds=None).fit(shuffle_latent_vectors)

In [97]:
import umap
act_reducer = umap.UMAP(n_neighbors=15,min_dist=0.1,metric="cosine",metric_kwds=None).fit(act_shuffle)

<p align="center">
    General Embeddings
</p>

In [19]:
embedding_v = reducer.transform(latent_vectors)
#embedding_act = act_reducer.transform(labels_norm)


In [8]:
import torch
from torch import nn
class Probe(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.probe = nn.Sequential(
            nn.Linear(in_dim, 2)
        )

    def train(self, data_emb : torch.Tensor, labels : torch.Tensor, num_epochs, learning_rate : float = 0.001, batch_size : int = 32) -> None:
        criterion = nn.CrossEntropyLoss()

        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

        logger.info('Training the probe..')
        
        for epoch in range(num_epochs):
            logger.info(f'Epoch {epoch}')
            for i in range(0, len(data_emb), batch_size):
                batch_emb = data_emb[i:i+batch_size]
                batch_labels = labels[i:i+batch_size]

                outputs = self.probe(batch_emb)
                
                loss = criterion(outputs, batch_labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        logger.info("Done")

    def evaluate(self, data_emb : torch.Tensor, labels : torch.Tensor, batch_size : int = 32) -> float:
        all_labels = torch.tensor([]).long()
        preds = torch.tensor([]).long()
        for i in range(0, len(data_emb), batch_size):
            logger.info(f"Evaluating batch {i}//{len(data_emb)/batch_size}")
            batch_emb = data_emb[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            with torch.no_grad():
                logits = self.probe(batch_emb)
            _, predicted = torch.max(logits, dim=-1)

            if i == 0:
                preds = predicted
                all_labels = batch_labels
            else:
                preds = torch.cat((preds, predicted), dim=0)
                all_labels = torch.cat((all_labels, batch_labels), dim=0)
        correct = (preds == all_labels).sum().item()
        accuracy = correct / all_labels.shape[0]
        logger.info(f"Probe accuracy: {accuracy:.4f}")
        return accuracy

In [20]:
from sklearn.utils import shuffle
import logging 
logger = logging.getLogger(__name__)
logging.basicConfig(filename='myapp.log', level=logging.INFO)
logger.info('Started')

noise_std = 0  # adjust strength of noise

obj_binary = wall_binary
labels = torch.from_numpy(obj_binary).long()

vae_probe = Probe(latent_vectors.shape[1]).to(device)

vae_vectors_t = torch.from_numpy(latent_vectors).to(device) + torch.randn_like(torch.from_numpy(latent_vectors).to(device)) * noise_std

vae_probe.train(data_emb=vae_vectors_t.to(device), labels=labels.to(device),
                num_epochs=2)

accuracy_vae = vae_probe.evaluate(data_emb=vae_vectors_t.to(device), labels=labels.to(device))


print(f"Accuracy VAE: {accuracy_vae}")


Accuracy VAE: 0.9036581709145427


In [20]:
embedding_v = embedding_v

modalities = ['v']
modality_cmap = ListedColormap(['red', 'green', 'orange'])
all_embeddings = [embedding_v] 
x_min = min([emb[:, 0].min() for emb in all_embeddings])
x_max = max([emb[:, 0].max() for emb in all_embeddings])
y_min = min([emb[:, 1].min() for emb in all_embeddings])
y_max = max([emb[:, 1].max() for emb in all_embeddings])

x_range = x_max - x_min
y_range = y_max - y_min
padding = 0.05
x_limits = [x_min - padding * x_range, x_max + padding * x_range]
y_limits = [y_min - padding * y_range, y_max + padding * y_range]

actions = ['right-left', 'front-back', 'up-down', 'gripper']
n_actions = len(actions)
n_modalities = len(modalities)

n_cols = n_modalities
n_rows = n_actions + 3

fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(6 * n_cols + 2, 4 * n_rows))
plt.subplots_adjust(hspace=0.3, wspace=0.3, right=0.92)

###################################
# Top row: Modality plot centered #
###################################
#axes[0, 0].remove()
#axes[0, 2].remove()

axes[0, -1].axis('off')



(0.0, 1.0, 0.0, 1.0)

In [21]:
actions = ['right-left', 'front-back',  'gripper','goal','ball','objects']
n_actions = len(actions)
n_modalities = len(modalities)
print(attributes[(attributes[:,-1]==0),6])
###################################
# Randomize plotting order to avoid points covering each other
###################################
np.random.seed(42)
shuffle_idx = np.random.permutation(len(x_disp))
wall_shuf = wall[shuffle_idx]
goal_shuf = goal[shuffle_idx]
ball_shuf = ball[shuffle_idx]
# Create shuffled versions of all data
x_disp_shuf = x_disp[shuffle_idx]
y_disp_shuf = y_disp[shuffle_idx]
z_disp_shuf = z_disp[shuffle_idx]
gripper_shuf = gripper[shuffle_idx]
modalities = ['v']

color_map = {
    #"none": '#1f77b4',
    "all": '#ff7f0e',
    "wall_ball": '#2ca02c',
    "wall": '#d62728',
    "goal_ball": '#9467bd'
}

# Shuffle embeddings
#embedding_v_shuf = embedding_v[shuffle_idx]
#embedding_act_shuf = embedding_act[shuffle_idx]
###################################
# Define masks on shuffled data
###################################
x_mouvement = ((x_disp_shuf==1)|(x_disp_shuf==-1))
x_stop = (np.abs(x_disp_shuf)<0.1)

x_disp_binary = x_disp_shuf[x_mouvement]
right_left_labels = np.where(x_disp_binary<0, 1,0)

y_filter = (y_disp_shuf>=-0.1)
y_filtered= y_disp_shuf[y_filter]
front_back_labels = np.where((np.abs(y_filtered)<0.2), 0,1)

#wall_binary = np.where((wall_shuf==-10),0,1)
goal_binary = np.where((goal_shuf==-10),0,1)
#ball_binary = np.where((ball_shuf<0),0,1)
binary_map = ListedColormap(['red', 'green'])


for row, action in enumerate(actions):
    actual_row = row 

    ax = axes[actual_row, 0]

    embedding = embedding_v[shuffle_idx]
    if action == 'right-left':
            ax.scatter(
                embedding_v[:, 0],
                embedding_v[:, 1],
                c='grey',
                s=5,
                alpha=0.3
            )
            scatter = ax.scatter(
                embedding[:, 0],
                embedding[:, 1],
                c=x_disp_shuf,
                cmap='plasma',
                s=5,
                alpha=0.9
            )
            ax.set_title(f' Right-Left')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2 
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
            
    elif action == 'front-back':
        ax.scatter(
            embedding_v[:, 0],
            embedding_v[:, 1],
            c='grey',
            s=5,
            alpha=0.3
        )
        scatter = ax.scatter(
            embedding[y_filter, 0],
            embedding[y_filter, 1],
            c=y_disp_shuf[y_filter],
            cmap='plasma',
            s=5,
            alpha=0.9
        )
        ax.set_title(f'Front-Stop')

        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        cbar_height = max(0.02, pos.height * 0.6) 
        cbar_y = pos.y0 + (pos.height - cbar_height) / 2
        cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
        cbar = plt.colorbar(scatter, cax=cbar_ax)
    elif action == 'gripper':
        ax.scatter(
            embedding_v[:, 0],
            embedding_v[:, 1],
            c='grey',
            s=5,
            alpha=0.3
        )
        scatter = ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=gripper_shuf,
            cmap='plasma',
            s=5,
            alpha=0.7
        )
        ax.set_title(f'Gripper')
        

        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        cbar_height = max(0.02, pos.height * 0.6) 
        cbar_y = pos.y0 + (pos.height - cbar_height) / 2
        cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
        cbar = plt.colorbar(scatter, cax=cbar_ax)
    elif action == 'goal':
        scatter = ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=goal_binary[:],
            cmap=binary_map,
            s=5,
            alpha=0.9
        )
        ax.set_title(f'Goal')

        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        cbar_height = max(0.02, pos.height * 0.6) 
        cbar_y = pos.y0 + (pos.height - cbar_height) / 2
        cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
        cbar = plt.colorbar(scatter, cax=cbar_ax)
    elif action == 'ball':
        scatter = ax.scatter(
            embedding_v[(attributes[:,-1]!=-1), 0],
            embedding_v[(attributes[:,-1]!=-1), 1],
            c=wall_binary[(attributes[:,-1]!=-1)],
            s=5,
            alpha=0.9
        )
        ax.set_title(f'Ball')
    
        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        cbar_height = max(0.02, pos.height * 0.6) 
        cbar_y = pos.y0 + (pos.height - cbar_height) / 2
        cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
        cbar = plt.colorbar(scatter, cax=cbar_ax)
    elif action == 'wall':
        scatter = ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=wall_binary[:],
            cmap=binary_map,
            s=5,
            alpha=0.9
        )
        ax.set_title(f'Wall')

        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        cbar_height = max(0.02, pos.height * 0.6) 
        cbar_y = pos.y0 + (pos.height - cbar_height) / 2
        cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
        cbar = plt.colorbar(scatter, cax=cbar_ax)
    
    elif action == 'objects':
        for obj_label, color in color_map.items():
            mask = obj_code == obj_label
            ax.scatter(embedding_v[mask, 0],
            embedding_v[mask, 1],
            c=color,
            alpha=0.8,
            s=5,
            label=obj_label,
            )
        ax.set_title(f'Objects')

        axes[actual_row, -1].axis('off')
        pos = axes[actual_row, -1].get_position()
        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=8, label=obj_label)
                    for obj_label, color in color_map.items()]
        axes[actual_row, -1].legend(handles=handles, loc='center left', fontsize=9)
        
        
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)

plt.suptitle('UMAP Visualization - Modalities and Attributes', fontsize=20, fontweight='bold', y=0.98)
plt.savefig(f"graphs/vae/UMAP_dataset_now.png")
plt.close()

[0.0264397  0.03541659 0.02226524 ... 0.02602971 0.02586842 0.02463382]


In [ ]:
from sklearn import svm
model = svm.SVC(kernel='linear')
from sklearn.utils import shuffle
import numpy as np

obj_binary = ball_binary
print('ball_binary')
points = np.concatenate([latent_vectors[(obj_binary == 1)],latent_vectors[(obj_binary == 0)]],axis=0)
#for act use labels_norm instead of vae_vectors
labels = np.concatenate([np.ones(len(latent_vectors[(obj_binary == 1)])),np.zeros(len(latent_vectors[(obj_binary == 0)]))])
points, labels = shuffle(points, labels, random_state=42)
print("Fitting VAE model")
model.fit(points,labels)


#embedding_points = np.concatenate([embedding_wall,embedding_no_wall])

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = model.predict(points)
print(accuracy_score(labels, y_pred))
print(precision_score(labels, y_pred))
print(recall_score(labels, y_pred))
print(f1_score(labels, y_pred))
print(confusion_matrix(labels, y_pred))




ball_binary
Fitting VAE model


In [ ]:
# =============================================================================
# CONFIGURATION DICTIONARY - Edit masks here to control visualizations
# Based on histogram analysis of action distributions
# =============================================================================

# Each action row has: 'masks' (list of filter conditions), 'colors', 'labels', 'title'
# Use None for continuous colormap instead of discrete categories

# Define masks based on histogram peaks and action modes
action_config = {
    'Movement Direction': {
        'type': 'categorical',
        'categories': [
            #{'mask': (np.abs(x_disp) < 0.3) & (y_disp > 0.5), 'color': 'green', 'label': 'Push Forward'},
            #{'mask': (x_disp < -0.5) & (np.abs(y_disp) < 0.3), 'color': 'blue', 'label': 'Push Left'},
            #{'mask': (x_disp > 0.5) & (np.abs(y_disp) < 0.3), 'color': 'red', 'label': 'Push Right'},
            {'mask': (np.abs(x_disp) > 0.1) & (np.abs(y_disp) < 0.2), 'color': 'orange', 'label': 'Stationary'},
        ],
        'show_background': True,
        'title': 'Movement Direction (XY plane)',
    },
    'Diagonal Movements': {
        'type': 'categorical',
        'categories': [
            {'mask': (x_disp < -0.1) , 'color': 'purple', 'label': 'Diagonal Left-Forward'},
            {'mask': (x_disp > 0.), 'color': 'cyan', 'label': 'Diagonal Right-Forward'},
            #{'mask': (x_disp < -0.3) & (y_disp < -0.3), 'color': 'brown', 'label': 'Diagonal Left-Back'},
            #{'mask': (x_disp > 0.3) & (y_disp < -0.3), 'color': 'pink', 'label': 'Diagonal Right-Back'},
        ],
        'show_background': True,
        'title': 'Diagonal Movements',
    },
    'Vertical + Gripper': {
        'type': 'categorical',
        'categories': [
            #{'mask': (z_disp > 0.5) & (gripper > 0.5), 'color': 'green', 'label': 'Up + Grip Close'},
            #{'mask': (z_disp > 0.5) & (gripper < -0.2), 'color': 'blue', 'label': 'Up + Grip Open'},
            {'mask': (z_disp < -0.5) & (gripper > 0.5), 'color': 'red', 'label': 'Down + Grip Close'},
            #{'mask': (z_disp < -0.5) & (gripper < -0.2), 'color': 'orange', 'label': 'Down + Grip Open'},
        ],
        'show_background': True,
        'title': 'Vertical Movement + Gripper',
    },
    'X-Displacement': {
        'type': 'continuous',
        'mask': None,
        'color_by': x_disp,
        'cmap': 'coolwarm',
        'title': 'X Displacement (Left-Right)',
    },
    'Y-Displacement': {
        'type': 'continuous',
        'mask': None,
        'color_by': y_disp,
        'cmap': 'coolwarm',
        'title': 'Y Displacement (Front-Back)',
    },
    'Gripper State': {
        'type': 'continuous',
        'mask': None,
        'color_by': gripper,
        'cmap': 'plasma',
        'title': 'Gripper Opening',
    },
}

# =============================================================================
# PLOTTING CODE
# =============================================================================

n_actions = len(action_config)
n_modalities = len(modalities)
n_rows = n_actions + 1
n_cols = n_modalities
modalities = ['v', 'act']
fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(6 * n_cols + 2, 4 * n_rows))
plt.subplots_adjust(hspace=0.3, wspace=0.3, right=0.92)

# Top row: plain modality embeddings
for col, modality in enumerate(modalities):
    ax = axes[0, col]
    ax.set_title(f'UMAP - {modality.upper()} in ', fontsize=14, fontweight='bold')
    ax.scatter(embedding_v_[modality][:, 0], embedding_v_[modality][:, 1], s=5, c='steelblue', alpha=0.5)
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
axes[0, -1].axis('off')

# Action rows
for row, (action_name, config) in enumerate(action_config.items()):
    actual_row = row + 1
    
    for col, modality in enumerate(modalities):
        ax = axes[actual_row, col]
        embedding = embedding_v_[modality]
        
        if config['type'] == 'continuous':
            # Continuous colormap
            mask = config.get('mask')
            if mask is None:
                scatter = ax.scatter(embedding[:, 0], embedding[:, 1],
                                     c=config['color_by'], cmap=config['cmap'], s=5, alpha=0.8)
            else:
                scatter = ax.scatter(embedding[mask, 0], embedding[mask, 1],
                                     c=config['color_by'], cmap=config['cmap'], s=5, alpha=0.8)
            
            # Add colorbar on last column
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6)
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                plt.colorbar(scatter, cax=cbar_ax)
                
        elif config['type'] == 'categorical':
            # Categorical with multiple masks
            if config.get('show_background', True):
                ax.scatter(embedding[:, 0], embedding[:, 1], c='lightgrey', s=1, alpha=0.3)
            
            for cat in config['categories']:
                mask = cat['mask']
                ax.scatter(embedding[mask, 0], embedding[mask, 1],
                           c=cat['color'], s=5, alpha=0.8, label=cat['label'])
            
            # Add legend on last column
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                handles = [plt.Line2D([0], [0], marker='o', color='w', 
                           markerfacecolor=cat['color'], markersize=8, label=cat['label'])
                           for cat in config['categories']]
                axes[actual_row, -1].legend(handles=handles, loc='center left', fontsize=9)
        
        ax.set_title(f"{modality.upper()} - {config['title']}", fontsize=10)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)

plt.suptitle('UMAP: Action Dimensions Across Modalities', fontsize=18, fontweight='bold', y=0.98)
plt.savefig(f"graphs/{current_model}/UMAP_action_config_comparison.png", dpi=150)
plt.show()

# Print mask statistics
print("\n=== Mask Statistics ===")
for action_name, config in action_config.items():
    print(f"\n{action_name}:")
    if config['type'] == 'categorical':
        for cat in config['categories']:
            print(f"  {cat['label']}: {cat['mask'].sum()} samples ({100*cat['mask'].sum()/len(x_disp):.1f}%)")
    elif config.get('mask') is not None:
        print(f"  Filtered: {config['mask'].sum()} samples ({100*config['mask'].sum()/len(x_disp):.1f}%)")